# DAI Mission — Proposal Template
**Data & AI in Economics | TU Dortmund**

This notebook is your team's mission proposal. Fill in every section before submission. Once approved, you will extend this same notebook into your final deliverable.

> **Team size:** 2–3 students  
> **Deliverable:** This Jupyter Notebook (proposal → final submission in one file)


## 1. Team

| Role | Name | Student ID |
|------|------|------------|
| Lead   |Usama Saleem |282362|
| Member |Zi Zhang |281726|
| Member |Ekin Yuecesan |284467|


## 2. Mission Title & Research Question

**Title:** Does Leverage Cause Corporate Financial Distress? A Causal Machine Learning Analysis of U.S. Public Firms Using Compustat Financial Statements

**Research question:**  
Can machine learning models trained on Compustat accounting ratios predict corporate financial distress more accurately than the classical Altman Z-Score, and does high financial leverage *causally* drive firms into distress, or is leverage itself a symptom of an already deteriorating business?

**Why it matters:**  
Corporate financial distress destroys firm value, eliminates jobs, and destabilises credit markets. Banks, credit funds, and rating agencies invest heavily in early-warning models to identify vulnerable firms before distress materialises. The 55-year-old Altman Z-Score remains an industry standard despite the dramatic advances in machine learning since 1968. This project asks whether modern ML methods: Lasso, Random Forest, and neural networks, systematically outperform Altman when applied to the same financial statement data, and uses causal inference to determine whether leverage is a genuine driver of distress or merely a correlated symptom. The answer has direct policy implications: if leverage causes distress, regulators should cap it; if leverage is endogenous, capital requirements alone will not prevent default. This question sits at the intersection of financial economics, machine learning, and credit risk management.


## 3. Data

**Source(s):**  
*Name each dataset, its provider, URL or access method, and licence/terms of use.*
| Compustat Annual Fundamentals (North America) | S&P Global via WRDS | WRDS web query → CSV download; TU Dortmund institutional access | Academic research licence; data not to be redistributed |

**Unit of observation:** *What does one row represent?*
One row = one firm-year (a U.S. publicly listed company in a given fiscal year), covering 2000–2022. Approximately 150,000–200,000 firm-year observations after filtering.

**Key variables:**

| Variable | Type | Role (feature / target / instrument / ...) | Description |
|----------|------|---------------------------------------------|-------------|
| `distressed` | Binary (0/1) | **Target** | 1 if firm has negative book equity (ceq < 0) OR interest coverage < 1 (ebitda/xint < 1) in the following year |
| `leverage` | Continuous | Feature + causal variable of interest | Long-term debt / Total assets (dltt / at) |
| `interest_coverage` | Continuous | Feature | EBITDA / Interest expense (ebitda / xint) |
| `roa` | Continuous | Feature | Net income / Total assets (ni / at) - profitability |
| `current_ratio` | Continuous | Feature | Current assets / Current liabilities (act / lct) - liquidity |
| `asset_growth` | Continuous | Feature | Year-on-year growth in total assets |
| `profit_margin` | Continuous | Feature | Net income / Revenue (ni / revt) |
| `debt_service` | Continuous | Feature | (dltt + xint) / ebitda - debt burden |
| `z_score` | Continuous | Feature + Benchmark | Altman Z-Score - classical 5-factor model as baseline |
| `size` | Continuous | Confounder | Log of total assets - controls for firm size |
| `sich` | Categorical | Confounder | SIC industry code - industry fixed effects |

**Potential data quality issues:**  
- **Survivorship bias:** Compustat only contains listed firms; very early delistings may be missing. Noted as a limitation.
- **Class imbalance:** Financial distress affects ~5–8% of firm-years. We use `class_weight='balanced'` and report ROC-AUC and Recall rather than accuracy to handle this properly.
- **Missing values:** Interest coverage undefined when xint = 0. Imputed with large positive value (99) reflecting low distress risk for debt-free firms.
- **Outliers:** Extreme ratio values (e.g. leverage > 10) winsorised at 1st and 99th percentile to reduce influence of data errors.
- **Look-ahead bias:** All features lagged one year relative to the distress label — a firm's 2019 ratios predict its 2020 distress status.


In [ ]:
# Data loading & first inspection
import pandas as pd
import numpy as np

# Load Compustat data downloaded from WRDS
df = pd.read_csv('./data/compustat_annual.csv')

df.head()
df.info()
df.describe()

# Construct distress label
df['distressed'] = ((df['ceq'] < 0) |
                    (df['ebitda'] / df['xint'].replace(0, np.nan) < 1)
                   ).astype(int)

# Check class balance
print(df['distressed'].value_counts(normalize=True))
print(f"Total observations: {len(df):,}")
print(f"Distressed firms: {df['distressed'].sum():,}")



## 4. Planned Methods

Your mission **must** apply at least one technique from **each** of the three blocks below. Tick the ones you plan to use and briefly justify the choice.

### 4a. Causal Inference
- [x] **Causal graph / DAG (DoWhy)**
- [x] **Backdoor adjustment**
- [ ] Instrumental variable
- [ ] Propensity score stratification
- [ ] Other: ___

**Justification:**
We construct a DAG capturing the leverage endogeneity problem: declining revenues causes both higher leverage (distressed firms borrow to survive) AND higher default probability, making leverage an endogenous predictor rather than a clean causal driver. Using DoWhy we formalise this graph and apply backdoor adjustment, controlling for firm size (`size`) and industry (`sich`) as observed confounders. We demonstrate Simpson's paradox: high-leverage firms appear more distressed in the pooled sample, but within capital-intensive industries (utilities, telecom) the relationship weakens or reverses which shows that the naive correlation is confounded by industry structure. Refutation tests validate that our causal estimates are robust. This section directly applies Lecture 02 (potential outcomes framework, DAGs, confounder identification) to a concrete financial problem.


- [x] **Linear / Ridge / Lasso regression**
- [x] **Logistic regression**
- [ ] k-Nearest Neighbors
- [x] **Support Vector Machine**
- [x] **Decision Tree / Random Forest**
- [x] **Neural network (regression or classification)**
- [ ] Other: ___

**Justification:**
Full supervised pipeline applied to binary distress prediction:
- **Lasso** for automatic feature selection from 20+ engineered financial ratios. It identifies which ratios survive vs. Altman's classical five. Post-Lasso OLS run on surviving features for unbiased coefficient estimates.
- **Logistic regression** as interpretable baseline: coefficients directly interpretable as log-odds of distress, useful for communicating results to non-technical stakeholders.
- **SVM (RBF kernel)** captures non-linear ratio interactions that logistic regression misses (e.g. high leverage is only dangerous when combined with low interest coverage AND declining revenues simultaneously).
- **Random Forest** expected best performer on structured tabular data; feature importances produce a data-driven ranking of which ratios matter most, directly comparable to Altman's hand-selected factors.
- **Neural network (Keras)** - two hidden layers, ReLU activation, dropout for regularisation, early stopping, sigmoid output for binary classification. Demonstrates overfit-then-regularise strategy from Lecture 05. Compared against Random Forest to assess whether added complexity is worth it on purely tabular data.

### 4c. Unsupervised Learning / Generative Models
- [x] **K-Means clustering**
- [x] **Hierarchical clustering**
- [ ] Variational autoencoder
- [ ] GAN
- [ ] Other: ___

**Justification:**
PCA applied to the full set of financial ratios, reducing to 2–3 principal components for visualisation and clustering. K-Means clustering (k chosen via elbow plot + silhouette score following Lecture 04) segments firms into natural financial health profiles without using labels e.g. "healthy cash-generative," "leveraged but profitable," "distressed and illiquid." Distress labels overlaid on PC1–PC2 scatter plot shows that unsupervised structure aligns strongly with the supervised prediction signal, validating both approaches. Hierarchical clustering with Ward linkage run as a comparison and dendrogram produced. Default rate computed per K-Means cluster, demonstrating how unsupervised segmentation can be used for portfolio-level credit risk monitoring without requiring labelled training data.


## 5. Evaluation Strategy

*How will you know if your mission succeeded? Describe:*

- The metric(s) you will use for each model (e.g. RMSE, accuracy, AUC, silhouette score).
- How you will validate causal claims (e.g. refutation tests, sensitivity analysis).
- Any baselines or benchmarks you will compare against.

**Metrics per model:**

| Model | Primary Metric | Secondary Metrics | Reason |
|-------|---------------|-------------------|--------|
| Lasso | Non-zero coefficients | Test R² vs Ridge | Sparsity = interpretability |
| Logistic Regression | ROC-AUC | Recall, Precision, F1 | AUC threshold-independent |
| SVM | ROC-AUC | Recall @ business threshold | Missed defaults most costly |
| Random Forest | ROC-AUC | Feature importance ranking | Compare vs Altman factors |
| Neural Network | ROC-AUC | Train vs val loss curve | Check overfitting |
| K-Means | Silhouette score | Default rate per cluster | Cluster meaningfulness |

**Baseline / benchmark:**
Altman Z-Score (1968), firm predicted distressed if Z < 1.81. ROC-AUC of the Z-Score computed on our test set and used as the primary benchmark against all ML models. This is the most well-known distress prediction model in finance, outperforming it is a meaningful, verifiable result.

**Causal claim validation:**
DoWhy refutation tests — `random_common_cause` (add a random variable as confounder, check estimate stability) and `placebo_treatment_refuter` (replace leverage with random noise, check effect disappears). E-value computed to quantify sensitivity to unobserved confounding.

**Train/test split:**
70% train, 30% test, split by year to respect temporal structure (no future data leaking into training). Cross-validation uses `TimeSeriesSplit(n_splits=5)`. All hyperparameters tuned on validation set only.



## 6. Work Plan

| Step | Owner | Description |
|------|-------|-------------|
| 1 | Usama | WRDS Compustat data cleaning, distress label construction, 20+ financial ratio engineering |
| 2 | Ekin | EDA : distributions, class imbalance analysis, Simpson's paradox demonstration, correlation heatmap |
| 3 | Zi Zhang | Causal inference block : DAG (DoWhy), backdoor adjustment, confounder analysis, refutation tests |
| 4 | Ekin | Supervised learning : Lasso, Logistic, SVM, Random Forest; ROC curves; model comparison table vs Altman |
| 5 |  Usama | Unsupervised block : PCA scree plot, K-Means elbow + silhouette, hierarchical dendrogram, cluster analysis |
| 6 | Usama | Neural network : Keras model, learning curves, final model comparison, overfit-then-regularise demonstration |
| 7 | Zi Zhang | Synthesis & write-up : Discussion, limitations, conclusion, notebook clean-up and submission |


---
## 7. Results *(complete for final submission)*


### 7a. Causal Inference

In [ ]:
# Causal inference analysis

### 7b. Supervised Learning

In [ ]:
# Supervised learning analysis

### 7c. Unsupervised / Generative

In [ ]:
# Unsupervised / generative analysis

## 8. Discussion & Conclusion *(complete for final submission)*

*Synthesise findings across all three method blocks. What does each lens reveal that the others miss? What are the limitations of your analysis?*
